## Recurrent Neural Networks

For recurrent network architectures, `keras` provides basic types such as `LSTM` networks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(12345)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.layers import Embedding
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence

### Data Preprocessing

The IMDB dataset contains movie review texts that are classified by sentiment as positive or negative. The data includes 25,000 examples for training and 25,000 for testing. When loading the data, we limit the vocabulary to the 5,000 most frequent words (all other words in the text will be marked as unknown).

In [ ]:
num_words = 5000
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=num_words)
print(x_train.shape, x_test.shape)

The data is preprocessed as sequences of indices corresponding to individual words in the text. The indices are assigned based on word frequency (i.e., index 1 represents the word that appears most frequently in the texts). Index 0 represents unknown/truncated words.

In [ ]:
print(x_train[0]) # print first training example

`keras` requires sequences to be aligned to the same length by adding 0 indices (so-called *padding*). To perform this alignment, you can use the helper function `sequence.pad_sequences`. By default, zeros are added at the beginning of the sequence. We set the maximum sequence length to 500.

In [ ]:
max_review_length = 500
x_train = sequence.pad_sequences(x_train, maxlen=max_review_length) # align training data
x_test = sequence.pad_sequences(x_test, maxlen=max_review_length)   # align test data

print(x_train[0])      # print the aligned first training example
print(len(x_train[0])) # the size of all vectors representing sequences will be 500

### Model creation

In [ ]:
# each word will be represented as a vector (so-called embedding); we set the size of the word representation to 32
# i.e., each input example (text) will be represented as a 500×32 matrix (maximum sequence length after padding ×
# size of the embedding vectors)
embedding_length = 32

model = Sequential()
# The Embedding layer creates a lookup table between a word index and its vector. Initially, the vectors
# for individual words are represented randomly; this layer transforms the input padded sequence of word indices
# into a matrix representing the example using embedding vectors for each word
model.add(Embedding(num_words, embedding_length, input_length=max_review_length))
# we create a recurrent LSTM layer (the size of the state and memory is 100)
model.add(LSTM(100))
# the final classification is calculated by a dense layer, which has the last LSTM state as input and one
# neuron with a sigmoid (logistic) activation function for binary classification
model.add(Dense(1, activation='sigmoid'))

### Training and Validation on Test Data

In [ ]:
# standard configuration of the optimization method for binary classification
adam = Adam()
model.compile(loss='binary_crossentropy', optimizer=adam, metrics=['accuracy'])

In [ ]:
# we train the model; as the validation set, we use the test data directly
# (i.e., after validation we obtain the final model accuracy directly)
f = model.fit(x_train, y_train, epochs=3, batch_size=128, validation_data=(x_test, y_test))